# Milestone 1: Data Understanding & Baseline Segmentation

## 1. Setup

In [1]:
import os
import numpy as np
import pandas as pd
import cv2

DATA_ROOT = '/kaggle/input/competitions/industrial-surface-defect-detection/public'
H, W = 128, 800
N = H * W
CLASSES = ['defect_a', 'defect_b', 'defect_c', 'defect_d']

df = pd.read_csv(f'{DATA_ROOT}/train.csv')
df['mask_rle'] = df['mask_rle'].fillna('').astype(str)
df['has'] = df['mask_rle'].str.strip() != ''
pos = df[df['has']]
n_images = df['image_id'].nunique()

assert len(df) == n_images * 4
print('rows:', len(df), '| images:', n_images, '| positive rows:', len(pos))

rows: 45572 | images: 11393 | positive rows: 7201


In [2]:
def rle_decode(rle, shape=(H, W)):
    mask = np.zeros(N, dtype=np.uint8)
    if rle.strip():
        t = list(map(int, rle.split()))
        for start, length in zip(t[0::2], t[1::2]):
            a = start - 1
            mask[a:min(a + length, N)] = 1
    return mask.reshape(shape, order='F')


def dice(x, y):
    sx, sy = x.sum(), y.sum()
    if sx + sy == 0:
        return 1.0
    return 2 * np.logical_and(x, y).sum() / (sx + sy)


def competition_score(dice_pos, fp_frac_empty, empty_correct):
    fg = float(np.mean(dice_pos)) if len(dice_pos) else 0.0
    fps = 1 - float(np.mean(fp_frac_empty)) if len(fp_frac_empty) else 1.0
    ea = float(np.mean(empty_correct)) if len(empty_correct) else 1.0
    return fg, fps, ea, 0.7 * fg + 0.2 * fps + 0.1 * ea

## 2. Question 1: Global Class Distribution & Clean Sample Mass

In [3]:
class_counts = pos['defect_type'].value_counts()
n_clean = n_images - pos['image_id'].nunique()

print(class_counts)
print('clean images:', n_clean)
print('Q1:', int(class_counts.max() + n_clean))

defect_type
defect_c    4100
defect_d    1707
defect_a    1081
defect_b     313
Name: count, dtype: int64
clean images: 5516
Q1: 9616


## 3. Question 2: Multi-Label Cardinality & Co-occurrence

In [4]:
card = pos.groupby('image_id')['defect_type'].nunique()

print(card.value_counts().sort_index())
print('Q2:', int((card > 1).sum()))

defect_type
1    4679
2    1072
3     126
Name: count, dtype: int64
Q2: 1198


## 4. Question 3: Spatial Mutual-Exclusivity Proof

In [5]:
multi_ids = card[card > 1].index
total_overlap = 0

for img_id, g in pos[pos['image_id'].isin(multi_ids)].groupby('image_id'):
    stacked = sum(rle_decode(r).astype(int) for r in g['mask_rle'])
    total_overlap += int((stacked >= 2).sum())

print('Q3:', total_overlap)

Q3: 0


## 5. Question 4: RLE Standards & Boundary Edge Case

In [6]:
def max_end(rle):
    t = list(map(int, rle.split()))
    return max(s + l - 1 for s, l in zip(t[0::2], t[1::2]))

ends = pos['mask_rle'].map(max_end)
n_bad = int((ends > N).sum())

print(f'Q4: {n_bad} rows, max index {int(ends.max())}')

Q4: 0 rows, max index 102400


## 6. Question 5: Geometric Distortion & Anisotropic Aspect Ratio

In [7]:
s_x = 256 / 800
s_y = 256 / 128

print(f'Q5: {s_x}, {s_y}, {s_y / s_x}')

Q5: 0.32, 2.0, 6.25


## 7. Question 6: Photometric Invisibility (Cohen's d)

In [8]:
IMG_EXT = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
img_paths = {}
for root, _, files in os.walk(DATA_ROOT):
    for f in files:
        if f.lower().endswith(IMG_EXT):
            img_paths[os.path.splitext(f)[0]] = os.path.join(root, f)

print('images found:', len(img_paths))

images found: 14393


In [9]:
acc = {c: np.zeros(6) for c in CLASSES}
skipped = 0

for r in pos.itertuples():
    path = img_paths.get(r.image_id)
    if path is None:
        skipped += 1
        continue
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None or img.shape != (H, W):
        skipped += 1
        continue
    img = img.astype(np.float64)
    m = rle_decode(r.mask_rle).astype(bool)
    d_px, b_px = img[m], img[~m]
    acc[r.defect_type] += [d_px.size, d_px.sum(), (d_px ** 2).sum(),
                           b_px.size, b_px.sum(), (b_px ** 2).sum()]

cohen = {}
for c, (nd, sd, qd, nb, sb, qb) in acc.items():
    md_, mb_ = sd / nd, sb / nb
    vd = (qd - nd * md_ ** 2) / (nd - 1)
    vb = (qb - nb * mb_ ** 2) / (nb - 1)
    pooled = np.sqrt(((nd - 1) * vd + (nb - 1) * vb) / (nd + nb - 2))
    cohen[c] = (md_ - mb_) / pooled

print('skipped rows:', skipped)
print(pd.Series(cohen).round(4))
invisible = [c for c, d in cohen.items() if abs(d) <= 0.02]
print('Q6:', ', '.join(f'{c}, texture/frequency' for c in invisible))

skipped rows: 0
defect_a    0.3198
defect_b    0.5317
defect_c    0.3683
defect_d   -0.0457
dtype: float64
Q6: 


## 8. Question 7: Metric Behavior of Naive Dice

In [10]:
gt = np.zeros((H, W), dtype=bool)
model_1 = np.zeros((H, W), dtype=bool)
model_2 = np.zeros((H, W), dtype=bool)
model_2[0, :2] = True

print(f'Q7: {dice(model_1, gt)}, {dice(model_2, gt)}')

Q7: 1.0, 0.0


## 9. Question 8: Official Competition Weighted Metric (Toy Batch)

In [11]:
dice_pos = np.array([0.8] * 6 + [0.4] * 4)
fp_frac = np.array([0.0] * 36 + [1024 / N] * 4)
empty_ok = np.array([True] * 36 + [False] * 4)

fg, fps, ea, score = competition_score(dice_pos, fp_frac, empty_ok)
print(f'Q8: {round(fg, 4)}, {round(fps, 4)}, {round(ea, 4)}, {round(score, 4)}')

Q8: 0.64, 0.999, 0.9, 0.7378


## 10. Question 9: All-Empty Baseline

In [12]:
n_empty = int((~df['has']).sum())
n_pos = int(df['has'].sum())

naive = n_empty / len(df)
_, _, _, official = competition_score(np.zeros(n_pos), np.zeros(n_empty), np.ones(n_empty, dtype=bool))

print('empty pairs:', n_empty)
print(f'Q9: {round(naive, 4)}, {round(official, 4)}')

empty pairs: 38371
Q9: 0.842, 0.3


## 11. Question 10: Morphological Noise Filtering

In [13]:
def component_areas(defect):
    areas = []
    for rle in pos.loc[pos['defect_type'] == defect, 'mask_rle']:
        m = rle_decode(rle)
        n, _, st, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
        areas.extend(st[1:, cv2.CC_STAT_AREA].tolist())
    return np.array(areas)

def lost_pct(areas, n_min):
    return 100 * areas[areas < n_min].sum() / areas.sum()

areas_a = component_areas('defect_a')
areas_b = component_areas('defect_b')

print(f'Q10: {int(areas_b.min())}, {round(lost_pct(areas_b, 50), 2)}, {round(lost_pct(areas_a, 200), 2)}')

Q10: 93, 0.0, 15.22


## 12. Question 11: Multi-Label Combination Stratification

In [14]:
labels = pos.groupby('image_id')['defect_type'].apply(lambda s: '+'.join(sorted(s)))
combo = labels.reindex(df['image_id'].drop_duplicates()).fillna('clean')
strata = combo.value_counts()
b_in_val = sum(0.2 * n for c, n in strata.items() if 'defect_b' in c)

print(strata)
print(f'Q11: {combo.nunique()}, {round(b_in_val)}')

defect_type
clean                         5516
defect_c                      3105
defect_d                       822
defect_a                       601
defect_c+defect_d              598
defect_a+defect_c              215
defect_b                       151
defect_a+defect_d              136
defect_a+defect_c+defect_d      87
defect_b+defect_c               67
defect_b+defect_d               33
defect_a+defect_b               23
defect_b+defect_c+defect_d      20
defect_a+defect_b+defect_d      11
defect_a+defect_b+defect_c       8
Name: count, dtype: int64
Q11: 15, 63


## 13. Question 12: Baseline Model & Decision Threshold Sweep

In [15]:
%pip install -q segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 1.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [16]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split

SEED = 42
EPOCHS = 8
BATCH = 16
THRESHOLDS = [0.3, 0.5, 0.7, 0.9]
device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(SEED)
np.random.seed(SEED)

rle_map = {(r.image_id, r.defect_type): r.mask_rle for r in df.itertuples()}
train_ids, val_ids = train_test_split(combo.index.to_numpy(), test_size=0.2, stratify=combo.values, random_state=SEED)

print('train:', len(train_ids), '| val:', len(val_ids), '| device:', device)

train: 9114 | val: 2279 | device: cuda


In [17]:
class SteelDataset(Dataset):
    def __init__(self, ids):
        self.ids = list(ids)

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        iid = self.ids[i]
        img = cv2.imread(img_paths[iid], cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        img = (img - 0.5) / 0.5
        masks = np.stack([rle_decode(rle_map[(iid, c)]) for c in CLASSES]).astype(np.float32)
        return torch.from_numpy(img)[None], torch.from_numpy(masks)


train_loader = DataLoader(SteelDataset(train_ids), batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(SteelDataset(val_ids), batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

In [18]:
model = smp.Unet('resnet34', encoder_weights=None, in_channels=1, classes=4).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scaler = torch.cuda.amp.GradScaler(enabled=device == 'cuda')
bce = nn.BCEWithLogitsLoss()


def dice_loss(logits, target, eps=1e-6):
    p = torch.sigmoid(logits)
    inter = (p * target).sum((2, 3))
    den = p.sum((2, 3)) + target.sum((2, 3))
    return 1 - ((2 * inter + eps) / (den + eps)).mean()


for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with torch.autocast(device_type=device, enabled=device == 'cuda'):
            logits = model(x)
            loss = bce(logits, y) + dice_loss(logits.float(), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running += loss.item() * x.size(0)
    print(f'epoch {epoch + 1}/{EPOCHS} | train loss {running / len(train_ids):.4f}')

/tmp/ipykernel_23/1836621350.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=device == 'cuda')


epoch 1/8 | train loss 1.0261
epoch 2/8 | train loss 0.9899
epoch 3/8 | train loss 0.9802
epoch 4/8 | train loss 0.9736
epoch 5/8 | train loss 0.9680
epoch 6/8 | train loss 0.9637
epoch 7/8 | train loss 0.9601
epoch 8/8 | train loss 0.9563


In [19]:
rec = {t: {'dice': [], 'gt_area': [], 'pred_area': []} for t in THRESHOLDS}

model.eval()
with torch.no_grad():
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        prob = torch.sigmoid(model(x))
        gt_area = y.sum((2, 3))
        for t in THRESHOLDS:
            pred = (prob > t).float()
            pred_area = pred.sum((2, 3))
            inter = (pred * y).sum((2, 3))
            den = pred_area + gt_area
            d = torch.where(den > 0, 2 * inter / den.clamp(min=1), torch.ones_like(den))
            rec[t]['dice'].append(d.flatten().cpu().numpy())
            rec[t]['gt_area'].append(gt_area.flatten().cpu().numpy())
            rec[t]['pred_area'].append(pred_area.flatten().cpu().numpy())

rows = []
for t in THRESHOLDS:
    d = np.concatenate(rec[t]['dice'])
    ga = np.concatenate(rec[t]['gt_area'])
    pa = np.concatenate(rec[t]['pred_area'])
    is_pos = ga > 0
    is_empty = ~is_pos
    fg, fps, ea, score = competition_score(d[is_pos], pa[is_empty] / N, pa[is_empty] == 0)
    rows.append({
        'threshold': t,
        'fp_rate_empty_pairs': float((pa[is_empty] > 0).mean()),
        'foreground_dice': fg,
        'naive_mean_dice': float(d.mean()),
        'official_score': score,
    })

results = pd.DataFrame(rows).set_index('threshold')
print(results.round(4))
print('holdout empty-pair fraction:', round(float(is_empty.mean()), 4))
print('peak threshold (naive mean Dice):', results['naive_mean_dice'].idxmax())
print('peak threshold (official score):', results['official_score'].idxmax())

           fp_rate_empty_pairs  foreground_dice  naive_mean_dice  \
threshold                                                          
0.3                     0.6869           0.4894           0.3410   
0.5                     0.6027           0.4743           0.4094   
0.7                     0.4775           0.4429           0.5099   
0.9                     0.2272           0.3292           0.7026   

           official_score  
threshold                  
0.3                0.5727  
0.5                0.5709  
0.7                0.5618  
0.9                0.5076  
holdout empty-pair fraction: 0.8419
peak threshold (naive mean Dice): 0.9
peak threshold (official score): 0.3
